In [69]:
import numpy as np
import pandas as pd 
import seaborn as sns 
import matplotlib.pyplot as plt
import warnings 
warnings.filterwarnings('ignore')


In [70]:
data= pd.read_csv('train.csv')
test= pd.read_csv('test.csv')
test_ids= test['PassengerId']

In [71]:
data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [72]:
data.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [73]:
def clean(data):
    data= data.drop(['Cabin','Ticket','Name','PassengerId'], axis =1)
    cols= ['Age','SibSp','Parch','Fare']
    for col in cols:
        data[col].fillna(data[col].median(), inplace = True)
    data['Embarked'].fillna("U", inplace = True)
    return data
data=clean(data)
test=clean(test)

data.head()


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


In [74]:
data['Embarked'].value_counts()

Embarked
S    644
C    168
Q     77
U      2
Name: count, dtype: int64

In [75]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
cols=['Sex', 'Embarked']
for col in cols:
    data[col]= le.fit_transform(data[col])
    test[col]= le.transform(test[col])

data.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,1,22.0,1,0,7.2500,2
1,1,1,0,38.0,1,0,71.2833,0
2,1,3,0,26.0,0,0,7.9250,2
3,1,1,0,35.0,1,0,53.1000,2
4,0,3,1,35.0,0,0,8.0500,2


In [76]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
x= data.drop('Survived',axis=1)
y=data['Survived']
X_train,X_test, y_train,  y_test= train_test_split(x,y,test_size=0.2,random_state=42)

In [77]:
y_train

331    0
733    0
382    0
704    0
813    0
      ..
106    1
270    0
860    0
435    1
102    0
Name: Survived, Length: 712, dtype: int64

In [78]:
clf= LogisticRegression(random_state=0,max_iter=1000).fit(X_train,y_train)

In [79]:
y_pred=clf.predict(X_test)

In [80]:
from sklearn.metrics import accuracy_score

In [81]:
accuracy_score(y_pred, y_test)

0.8100558659217877

In [82]:
submission_preds= clf.predict(test)

In [83]:
df= pd.DataFrame({
    'PassengerId':test_ids.values,
    'Survived':submission_preds,
})

In [84]:
df.to_csv('submission.csv', index=False)